# IndicConformer-600M-Multilingual — Kathbath (Vistaar) Test-Set Evaluation

Evaluates `ai4bharat/indic-conformer-600m-multilingual` (CTC + RNNT decoding) on the
**Kathbath** benchmark test set (via the official Vistaar release) - starting with
**Hindi**, with Bengali, Tamil, Marathi, and Telugu ready to add.

**No manual dataset upload needed** - the notebook downloads the official combined
benchmark zip directly from AI4Bharat's object store and extracts only the languages you
need. Each language folder has a `manifest.json` (NeMo-style JSONL: one
`{"audio_filepath", "duration", "text"}` object per line) and a `wavs/` folder of audio.

**Before running:**
1. Accept the gated model terms at
   https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual and set your HF
   token in the login cell (the dataset itself is public, no HF access needed for it).
2. Run cells top to bottom - the download/extract cell handles Hindi automatically.


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))


In [2]:
!nvidia-smi


Tue Jul 21 05:56:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q transformers torchaudio "onnxruntime==1.20.1" "onnx==1.20.1"
!pip install -q jiwer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 78.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 84.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.

## Download the Kathbath (Vistaar) benchmark automatically

No manual upload needed. AI4Bharat hosts the Vistaar benchmark test sets as a single combined
zip covering all 12 languages: `kathbath.zip` from
https://github.com/AI4Bharat/vistaar (see "Download Training Datasets and Benchmarks").

The zip's internal layout matches exactly what you already had working locally -
`kathbath/<lang>/manifest.json` + `kathbath/<lang>/wavs/*.wav` - so no manifest editing is
needed here either.

Since it's one zip for all languages, the cell below downloads the whole archive once
(cached - re-running skips re-download), then **selectively extracts only the languages
listed in `LANGS_TO_EXTRACT`** so you're not paying disk/time for languages you're not
using yet. Starting with Hindi only - add more language names to the list and re-run this
cell whenever you want to bring in Bengali, Tamil, Marathi, Telugu, etc.


In [4]:
import os

ZIP_URL = "https://indicwhisper.objectstore.e2enetworks.net/vistaar_benchmarks/kathbath.zip"
DOWNLOAD_DIR = "/kaggle/working/kathbath_download"
ZIP_PATH = os.path.join(DOWNLOAD_DIR, "kathbath.zip")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)


def zip_is_valid(path):
    """Returns True only if the file exists AND is a complete, openable zip archive.
    A partial/interrupted download can still have a valid-looking header, so we
    actually try to open it and read the central directory."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return False
    try:
        import zipfile
        with zipfile.ZipFile(path, "r") as zf:
            bad_file = zf.testzip()  # returns None if all entries check out
            return bad_file is None
    except Exception:
        return False


if zip_is_valid(ZIP_PATH):
    print(f"Zip already downloaded and verified at {ZIP_PATH}, skipping download.")
else:
    if os.path.exists(ZIP_PATH):
        print(f"Found an existing file at {ZIP_PATH} but it's incomplete/corrupt "
              f"(size={os.path.getsize(ZIP_PATH) / 1e9:.2f} GB) - resuming download.")
    else:
        print("Downloading Kathbath (Vistaar benchmark) zip - single archive covering all 12")
        print("languages, so this can take a while depending on connection speed...")
    # -c resumes from where a partial file left off instead of restarting from zero
    !wget -c -O {ZIP_PATH} {ZIP_URL}

    if not zip_is_valid(ZIP_PATH):
        raise RuntimeError(
            f"Download finished but {ZIP_PATH} is still not a valid/complete zip "
            f"(size={os.path.getsize(ZIP_PATH) / 1e9:.2f} GB). This usually means the "
            f"session was interrupted again, or the server doesn't support resumable "
            f"range requests for this URL. Try re-running this cell (wget -c will keep "
            f"resuming), or delete {ZIP_PATH} to force a full fresh download."
        )
    print("Zip verified OK.")

print(f"Zip size: {os.path.getsize(ZIP_PATH) / 1e9:.2f} GB")


languages, so this can take a while depending on connection speed...
--2026-07-21 05:58:57--  https://indicwhisper.objectstore.e2enetworks.net/vistaar_benchmarks/kathbath.zip
Resolving indicwhisper.objectstore.e2enetworks.net (indicwhisper.objectstore.e2enetworks.net)... 164.52.206.155, 164.52.206.154, 101.53.152.33, ...
Connecting to indicwhisper.objectstore.e2enetworks.net (indicwhisper.objectstore.e2enetworks.net)|164.52.206.155|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3457654951 (3.2G) [application/zip]
Saving to: ‘/kaggle/working/kathbath_download/kathbath.zip’

/kaggle/working/kat 100%[===================>]   3.22G  16.9MB/s    in 3m 25s  

2026-07-21 06:02:23 (16.1 MB/s) - ‘/kaggle/working/kathbath_download/kathbath.zip’ saved [3457654951/3457654951]

Zip verified OK.
Zip size: 3.46 GB


In [22]:
import zipfile
from tqdm import tqdm

EXTRACT_ROOT = "/kaggle/working/kathbath_data"
os.makedirs(EXTRACT_ROOT, exist_ok=True)

# ── Languages to extract right now. Add more later ("bengali", "tamil", "marathi",
# "telugu") and re-run this cell to bring in additional languages without re-downloading.
LANGS_TO_EXTRACT = ["marathi"]

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    all_names = zf.namelist()
    top_level = all_names[0].split("/")[0]
    print(f"Top-level folder inside zip: '{top_level}'")

    already_extracted = {
        lang for lang in LANGS_TO_EXTRACT
        if os.path.exists(os.path.join(EXTRACT_ROOT, top_level, lang, "manifest.json"))
    }
    to_process = [lang for lang in LANGS_TO_EXTRACT if lang not in already_extracted]
    if already_extracted:
        print(f"Already extracted, skipping: {sorted(already_extracted)}")
    if not to_process:
        print("Nothing new to extract.")
    else:
        to_extract = [
            n for n in all_names
            if any(n.startswith(f"{top_level}/{lang}/") for lang in to_process)
        ]
        print(f"Extracting {len(to_extract)} files for: {to_process} ...")
        for name in tqdm(to_extract):
            zf.extract(name, EXTRACT_ROOT)

KATHBATH_ROOT = os.path.join(EXTRACT_ROOT, top_level)
print(f"\nKATHBATH_ROOT = {KATHBATH_ROOT}")
print(f"Contents: {os.listdir(KATHBATH_ROOT)}")


Top-level folder inside zip: 'kathbath'
Already extracted, skipping: ['marathi']
Nothing new to extract.

KATHBATH_ROOT = /kaggle/working/kathbath_data/kathbath
Contents: ['marathi', 'telugu']


In [5]:
from huggingface_hub import login
import os

# SECURITY: don't hardcode HF tokens in the notebook. Use Kaggle's built-in
# "Add-ons > Secrets" (or an environment variable) so the token isn't stored in
# plain text in the .ipynb file, which is easy to accidentally share/commit.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")

login(token=hf_token)
print("Successfully logged into Hugging Face Hub!")


Successfully logged into Hugging Face Hub!


In [6]:
from transformers import AutoModel
import torch
import torchaudio

# 1. Load the model (once - reused across all 5 languages)
print("Loading the model... (this will take a few minutes on first run)")
model = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}")


Loading the model... (this will take a few minutes on first run)


config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual:
- model_onnx.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:115: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Model loaded on cuda


## Config

Point `KATHBATH_ROOT` at wherever your attached Kaggle Dataset lands - check the file
listing printed in the first cell, or the "Data" panel on the right, to confirm the exact
path (it's usually `/kaggle/input/<your-dataset-slug>`, which may or may not itself
contain a `kathbath/` subfolder depending on how you zipped it).

Each `KATHBATH_ROOT/<lang_folder>/` is expected to contain:
- `manifest.json` - one JSON object per line: `{"audio_filepath": ..., "duration": ..., "text": ...}`
- an audio subfolder (commonly `wav/` or `wavs/`) - the exact name doesn't matter, the
  notebook indexes every `.wav`/`.flac`/`.mp3` file under the language folder by filename.

Sample counts you reported (Vistaar-filtered Kathbath test sets):

| Language | Folder | Model lang code | Test samples |
|---|---|---|---|
| Hindi   | `hindi`   | `hi` | 1,929 |
| Bengali | `bengali` | `bn` | 1,783 |
| Tamil   | `tamil`   | `ta` | 1,642 |
| Marathi | `marathi` | `mr` | 1,631 |
| Telugu  | `telugu`  | `te` | 1,492 |

Total = 8,477 utterances x 2 decoding passes (CTC + RNNT) \u2248 16,954 inference calls -
comfortably more T4-friendly than the full raw Kathbath `valid` split, but still budget a
few hours. Checkpointing lets you resume across sessions if needed.


In [23]:
# ── KATHBATH_ROOT is already set by the download/extract cell above.
# If you're attaching a manually-uploaded Kaggle Dataset instead, uncomment and edit:
# KATHBATH_ROOT = "/kaggle/input/kathbath"

# (kathbath_folder_name, model_language_code, display_name)
# Only languages actually extracted above will work - add entries here as you add them
# to LANGS_TO_EXTRACT and re-run the extraction cell.
LANGUAGES = [
    # ("hindi",   "hi", "Hindi"),
    # ("bengali", "bn", "Bengali"),
    # ("tamil",   "ta", "Tamil"),
    ("marathi", "mr", "Marathi"),
    # ("telugu",  "te", "Telugu"),
]

# Set to an int (e.g. 300) to cap samples per language for a quick test run,
# or None to evaluate every sample in each language's manifest.json.
SAMPLES_PER_LANG = None

CHECKPOINT_EVERY = 100
CHECKPOINT_DIR = "/kaggle/working"


In [16]:
import json
import os
from pathlib import Path
import torch
import torchaudio
import numpy as np
from tqdm import tqdm
from jiwer import wer, cer


def read_manifest(manifest_path):
    """Reads a NeMo-style manifest: one JSON object per line. Falls back to a single
    JSON array if the file isn't line-delimited."""
    with open(manifest_path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    entries = []
    try:
        for line in content.splitlines():
            line = line.strip()
            if not line:
                continue
            entries.append(json.loads(line))
    except json.JSONDecodeError:
        entries = json.loads(content)
    return entries


def build_audio_index(lang_dir: Path) -> dict:
    """Scans lang_dir recursively once and maps basename -> full path. Robust to any
    audio subfolder naming ('wav', 'wavs', etc.) and any nesting depth."""
    index = {}
    for ext in ("*.wav", "*.flac", "*.mp3"):
        for p in lang_dir.rglob(ext):
            index[p.name] = str(p)
    return index


def resolve_audio_path(lang_dir: Path, audio_filepath: str, audio_index: dict) -> str:
    """audio_filepath in the manifest may be absolute (from the original download
    machine), relative to some root above lang_dir (e.g. 'kathbath/hindi/wavs/x.wav'),
    or just a bare filename. Try direct candidates first, then fall back to the
    prebuilt basename index (handles any subfolder naming mismatch, e.g. 'wav' vs 'wavs')."""
    p = Path(audio_filepath)
    candidates = [
        p if p.is_absolute() else None,
        lang_dir / audio_filepath,
        lang_dir.parent / audio_filepath,  # manifest path already includes '<lang>/wavs/...'
        lang_dir / "wav" / p.name,
        lang_dir / "wavs" / p.name,
        lang_dir / p.name,
    ]
    for c in candidates:
        if c is not None and c.exists():
            return str(c)
    if p.name in audio_index:
        return audio_index[p.name]
    raise FileNotFoundError(
        f"Could not resolve audio file '{audio_filepath}' under {lang_dir} "
        f"(tried direct path candidates and a full basename index of {len(audio_index)} "
        f"audio files under this folder)."
    )


def checkpoint_path(lang_folder):
    return os.path.join(CHECKPOINT_DIR, f"checkpoint_{lang_folder}.json")


def save_checkpoint(idx, refs, preds_ctc, preds_rnnt, lang_folder):
    path = checkpoint_path(lang_folder)
    data = {
        "last_index": idx,
        "references": refs,
        "predictions_ctc": preds_ctc,
        "predictions_rnnt": preds_rnnt,
    }
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)
    print(f"  \u2714 Checkpoint saved at sample {idx} \u2192 {path}")


def load_checkpoint(lang_folder):
    path = checkpoint_path(lang_folder)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Resuming {lang_folder} from checkpoint: {data['last_index'] + 1} samples already done.")
        return data["last_index"] + 1, data["references"], data["predictions_ctc"], data["predictions_rnnt"]
    return 0, [], [], []


def load_audio2(audio_source, target_sr=16000):
    waveform, original_sr = torchaudio.load(audio_source)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if original_sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=original_sr, new_freq=target_sr)
        waveform = resampler(waveform)
    return waveform


def evaluate_language(lang_folder, language_code, display_name, kathbath_root, n_samples=None):
    """Reads manifest.json for one language, runs CTC + RNNT inference over its audio,
    checkpointing periodically. Returns (references, predictions_ctc, predictions_rnnt)."""
    print(f"\n{'='*70}\nEvaluating: {display_name}  (folder={lang_folder}, lang_code={language_code})\n{'='*70}")

    lang_dir = Path(kathbath_root) / lang_folder
    manifest_path = lang_dir / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"manifest.json not found at {manifest_path} - check KATHBATH_ROOT.")

    entries = read_manifest(manifest_path)
    if n_samples is not None:
        entries = entries[:n_samples]
    total = len(entries)
    print(f"Found {total} entries in manifest.")

    print("Indexing audio files...")
    audio_index = build_audio_index(lang_dir)
    print(f"Indexed {len(audio_index)} audio files under {lang_dir}.")

    start_idx, references, predictions_ctc, predictions_rnnt = load_checkpoint(lang_folder)

    last_idx = start_idx - 1
    for sample_idx in tqdm(range(start_idx, total), initial=start_idx, total=total, desc=display_name):
        entry = entries[sample_idx]
        reference_text = entry["text"]

        try:
            wav_path = resolve_audio_path(lang_dir, entry["audio_filepath"], audio_index)
            audio_input = load_audio2(wav_path).to(device)

            with torch.no_grad():
                transcription_ctc = model(audio_input, language_code, "ctc")
                transcription_rnnt = model(audio_input, language_code, "rnnt")

            references.append(reference_text)
            predictions_ctc.append(transcription_ctc.strip())
            predictions_rnnt.append(transcription_rnnt.strip())
            last_idx = sample_idx

        except (RuntimeError, FileNotFoundError) as e:
            print(f"Skipping sample {sample_idx} due to error: {e}.")
            continue

        if (sample_idx + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(sample_idx, references, predictions_ctc, predictions_rnnt, lang_folder)

    save_checkpoint(last_idx, references, predictions_ctc, predictions_rnnt, lang_folder)
    print(f"Inference for {display_name} complete: {len(references)} samples evaluated.")
    return references, predictions_ctc, predictions_rnnt


In [24]:
# ── Run evaluation for all 5 languages ───────────────────────────────────────
results = {}

for lang_folder, language_code, display_name in LANGUAGES:
    refs, preds_ctc, preds_rnnt = evaluate_language(
        lang_folder, language_code, display_name, KATHBATH_ROOT, n_samples=SAMPLES_PER_LANG
    )
    results[display_name] = {
        "references": refs,
        "predictions_ctc": preds_ctc,
        "predictions_rnnt": preds_rnnt,
    }



Evaluating: Marathi  (folder=marathi, lang_code=mr)
Found 1631 entries in manifest.
Indexing audio files...
Indexed 1631 audio files under /kaggle/working/kathbath_data/kathbath/marathi.



Marathi:   6%|▌         | 100/1631 [02:59<43:24,  1.70s/it]

  ✔ Checkpoint saved at sample 99 → /kaggle/working/checkpoint_marathi.json



Marathi:  12%|█▏        | 200/1631 [05:58<40:16,  1.69s/it]

  ✔ Checkpoint saved at sample 199 → /kaggle/working/checkpoint_marathi.json


...

  ✔ Checkpoint saved at sample 1499 → /kaggle/working/checkpoint_marathi.json



Marathi:  98%|█████████▊| 1600/1631 [48:50<00:51,  1.66s/it]

  ✔ Checkpoint saved at sample 1599 → /kaggle/working/checkpoint_marathi.json



Marathi: 100%|██████████| 1631/1631 [49:51<00:00,  1.83s/it]

  ✔ Checkpoint saved at sample 1630 → /kaggle/working/checkpoint_marathi.json
Inference for Marathi complete: 1631 samples evaluated.


In [25]:
# ── Compute WER/CER per language and build a summary table ──────────────────
rows = []
for display_name, r in results.items():
    refs, preds_ctc, preds_rnnt = r["references"], r["predictions_ctc"], r["predictions_rnnt"]
    if not refs:
        print(f"No samples evaluated for {display_name}, skipping metrics.")
        continue
    ctc_wer, ctc_cer = wer(refs, preds_ctc), cer(refs, preds_ctc)
    rnnt_wer, rnnt_cer = wer(refs, preds_rnnt), cer(refs, preds_rnnt)
    rows.append({
        "Language": display_name,
        "N": len(refs),
        "CTC WER %": round(ctc_wer * 100, 2),
        "CTC CER %": round(ctc_cer * 100, 2),
        "RNNT WER %": round(rnnt_wer * 100, 2),
        "RNNT CER %": round(rnnt_cer * 100, 2),
    })

summary_df = pd.DataFrame(rows).set_index("Language")
print(summary_df.to_string())

out_csv = "/kaggle/working/indicconformer_kathbath_summary.csv"
summary_df.to_csv(out_csv)
print(f"\nSaved summary to {out_csv}")


             N  CTC WER %  CTC CER %  RNNT WER %  RNNT CER %
Language                                                    
Marathi   1631      17.61       5.62       17.14         5.5

Saved summary to /kaggle/working/indicconformer_kathbath_summary.csv


## Notes

- **Path issues:** if you get a `manifest.json not found` error, print
  `os.listdir(KATHBATH_ROOT)` to see the exact top-level folder name Kaggle mounted your
  dataset under, and adjust `KATHBATH_ROOT` accordingly (Kaggle sometimes nests an extra
  folder level depending on how the dataset was zipped/uploaded).
- **`audio_filepath` resolution:** manifests can have absolute paths from the original
  download machine, paths already prefixed with `kathbath/<lang>/wavs/...`, or bare
  filenames - all of which break naive joining against `KATHBATH_ROOT`. The notebook
  scans each language folder once (`build_audio_index`) and matches every manifest entry
  by filename, so it's robust regardless of subfolder naming (`wav` vs `wavs`) or path
  prefixes in the manifest - no manifest editing needed.
- **Resuming across sessions:** each language's `checkpoint_<lang>.json` in
  `/kaggle/working` holds every reference/prediction gathered so far. If a session times
  out, save `/kaggle/working` as a Kaggle Dataset, start a new session, copy the
  checkpoint files back into `/kaggle/working`, and re-run - it resumes from
  `last_index + 1` per language automatically.
- **CTC vs RNNT:** RNNT decoding (autoregressive beam search) is slower than CTC
  (greedy/frame-based), so most wall-clock time per language goes to the RNNT pass.
